# SEOBNRv5PHM Balance-Law Diagnostics

Generate a SEOBNRv5PHM waveform through the local EOB wrapper, compute chunked balance-law residuals, optionally add displacement memory, and visualize angular and spectral residual diagnostics.

In [ ]:
import numpy as np
from pprint import pprint
import matplotlib.pyplot as plt
from spectools.spherical.grids import GLGrid

from waveformtools.chunked_balance_laws import balance_law_chunked
from waveformtools.integrate import TwoDIntegral
from waveformtools.models.eob import EOBWaveformModel
from waveformtools.spherical_array import SphericalArray

In [ ]:
params = {
    "approximant": "SEOBNRv5PHM",
    "mass1": 40.0,
    "mass2": 35.0,
    "spin1x": 0.08,
    "spin1y": -0.03,
    "spin1z": 0.25,
    "spin2x": -0.04,
    "spin2y": 0.02,
    "spin2z": -0.15,
    "distance": 400.0,
    "inclination": 0.7,
    "phi_ref": 0.2,
    "f_lower": 15.0,
    "f_ref": 15.0,
    "f_max": 512.0,
    "delta_t": 1.0 / 4096.0,
    "delta_f": 1.0 / 4.0,
    "ell_max": 4,
}
grid = GLGrid(L=8)
chunk_size = 1024

In [ ]:
model = EOBWaveformModel(parameters_dict=params)
strain = model.get_td_waveform_modes(dimensionless=True, L=params["ell_max"])
strain._Grid = grid

mass1 = params["mass1"]
mass2 = params["mass2"]
total_mass = mass1 + mass2
mu = (mass1 / total_mass) * (mass2 / total_mass)
E0 = model.model.dynamics[0, 5] * mu

news = strain.get_news_from_strain()
Erad = strain.compute_energy_radiated(news_modes=news)
Mfinal = E0 - Erad
kick = strain.compute_kick(Mfinal=Mfinal)
print(f"n_times={strain.data_len}, ell_max={strain.ell_max}")
print(f"E0={E0:.12e}, Erad={Erad:.12e}, Mfinal={Mfinal:.12e}, kick={kick}")

In [ ]:
finite_time_diag = strain.diagnose_displacement_memory_finite_time(
    window_fraction=0.1,
)
omitted_inspiral_diag = strain.diagnose_omitted_inspiral(
    min_cycles=3.0,
    high_initial_power_fraction=0.05,
)
print("finite-time memory diagnostic")
pprint(finite_time_diag)
print("omitted-inspiral diagnostic")
pprint(omitted_inspiral_diag)

In [ ]:
def balance_residual(modes):
    residual, info = balance_law_chunked(
        strain_modes=modes,
        ginfo=grid,
        M_adm=E0,
        M_final=Mfinal,
        v_kick=kick,
        chunk_size=chunk_size,
        ell_max=modes.ell_max,
        debug=True,
    )
    rms = np.sqrt(np.real(TwoDIntegral(np.abs(residual) ** 2, grid)) / (4.0 * np.pi))
    return residual, info, float(rms)

residual, info, rms = balance_residual(strain)
with_memory = strain.with_displacement_memory()
with_memory._Grid = grid
residual_mem, info_mem, rms_mem = balance_residual(with_memory)
print(f"RMS residual original: {rms:.8e}")
print(f"RMS residual with memory: {rms_mem:.8e}")
print(f"ratio: {rms_mem / rms:.8e}")

In [ ]:
theta, phi = grid.meshgrid
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, data, title in [
    (axes[0], residual.real, "Re residual"),
    (axes[1], residual_mem.real, "Re residual + memory"),
    (axes[2], (residual_mem - residual).real, "memory change"),
]:
    mesh = ax.pcolormesh(phi, theta, data, shading="auto")
    ax.set_title(title)
    ax.set_xlabel(r"$\\phi$")
    ax.set_ylabel(r"$\\theta$")
    fig.colorbar(mesh, ax=ax)
plt.show()

In [ ]:
def residual_modes(residual_grid, ell_max=6):
    arr = SphericalArray(
        label="residual_time_domain",
        time_axis=np.array([0.0]),
        data=residual_grid[:, :, np.newaxis],
        data_len=1,
        Grid=grid,
        spin_weight=0,
        ell_max=ell_max,
    )
    return arr.to_modes_array(Grid=grid, spin_weight=0, ell_max=ell_max)

res_modes = residual_modes(residual, ell_max=6)
res_mem_modes = residual_modes(residual_mem, ell_max=6)
labels, vals, vals_mem = [], [], []
for ell, emms in res_modes.modes_list:
    for emm in emms:
        labels.append(f"{ell},{emm}")
        vals.append(abs(res_modes.mode(ell, emm)[0]))
        vals_mem.append(abs(res_mem_modes.mode(ell, emm)[0]))

x = np.arange(len(labels))
plt.figure(figsize=(14, 4))
plt.semilogy(x, vals, "o", label="original")
plt.semilogy(x, vals_mem, "x", label="with memory")
plt.xticks(x, labels, rotation=90)
plt.ylabel("|residual mode|")
plt.legend()
plt.tight_layout()
plt.show()
print("l00 original:", res_modes.mode(0, 0)[0])
print("l00 with memory:", res_mem_modes.mode(0, 0)[0])